# Laboratorio 22. Pronóstico de demanda: MLP, CNN 1D y LSTM frente a baselines

**Pregunta guía:** ¿Superan las redes neuronales a los pronósticos ingenuos para predecir la demanda diaria de la próxima semana, y qué arquitectura aprovecha mejor la historia y las promociones planificadas?

## Objetivos
Al finalizar podrás construir ventanas de entrenamiento con historia y covariables conocidas a futuro; particionar en el tiempo sin fuga de información; entrenar MLP, CNN 1D y LSTM con el mismo protocolo; compararlos con baselines estacionales usando MAE, RMSE y WAPE; y analizar el error por horizonte.

## Prerrequisitos y conexión
Usa recurrencia y LSTM (laboratorios 15 y 16), convoluciones (laboratorio 12) y *early stopping* (laboratorio 20). Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Un pronóstico directo multi-horizonte aprende $\hat{y}_{t:t+H}=f_\theta(y_{t-L:t},\,x_{t-L:t},\,x^{fut}_{t:t+H})$: a partir de las últimas $L$ observaciones y covariables, y de las covariables **conocidas a futuro** (promociones planificadas, feriados), predice los próximos $H$ días de una vez. La MLP trata la ventana como un vector plano; la CNN 1D aplica filtros a lo largo del tiempo (con dilatación para ampliar el campo receptivo); la LSTM procesa la ventana paso a paso y resume la historia en su estado oculto.

## Problema y datos
Una cadena ficticia necesita pronosticar la demanda diaria de un producto en 4 tiendas para planificar reposición semanal ($H=7$). Generamos 3 años con tendencia, estacionalidad semanal y anual, promociones (con caída al día siguiente), feriados y ruido de Poisson. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
El costo del error no es simétrico: faltar producto pierde ventas y sobrar inmoviliza capital. WAPE (error absoluto total sobre demanda total) se comunica bien a negocio. Las promociones futuras son un dato que pertenece a otra área: un buen pronóstico depende de que ese dato esté gobernado y disponible a tiempo.

## Experimento conceptual
¿Qué gana cada arquitectura sobre el pronóstico "igual que la semana pasada"? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar la demanda diaria
Cada componente del generador corresponde a un patrón que un modelo debería capturar.

In [ ]:
n_stores, n_days = 4, 3 * 365
days = np.arange(n_days)
store_scale = np.array([120.0, 80.0, 200.0, 60.0])
weekly = np.array([0.85, 0.90, 0.95, 1.00, 1.15, 1.35, 1.05])  # lunes a domingo
yearly = 1 + 0.20 * np.sin(2 * np.pi * (days - 250) / 365)
trend = 1 + 0.15 * days / n_days
holiday = np.zeros(n_days, dtype=bool)
holiday[[year * 365 + day for year in range(3) for day in (0, 120, 170, 260, 358)]] = True
promo = rng.random((n_stores, n_days)) < 0.08
after_promo = np.roll(promo, 1, axis=1) & ~promo
after_promo[:, 0] = False

expected = (store_scale[:, None] * trend * yearly * weekly[days % 7]
            * (1 + 0.6 * promo) * (1 - 0.15 * after_promo) * (1 - 0.5 * holiday))
demand = rng.poisson(expected).astype(np.float32)

fig, axis = plt.subplots(figsize=(10, 3))
for store in range(n_stores):
    axis.plot(days[:120], demand[store, :120], label=f'tienda {store}')
axis.set_xlabel('día')
axis.set_ylabel('unidades')
axis.set_title('Primeros 120 días de demanda')
axis.legend(ncol=4)
plt.show()
print('demanda media por tienda:', demand.mean(axis=1).round(1))

**Resultado e interpretación.** Se aprecian el ciclo semanal, los picos de promoción y las caídas en feriados. Las tiendas tienen escalas distintas: para entrenar un solo modelo con todas, normalizaremos cada serie por su nivel.

### Partición temporal y ventanas
Dividimos por tiempo: train (primer 70 %), validación (siguiente 15 %) y test (último 15 %). Una ventana pertenece al split donde caen los días que pronostica; su historia puede venir de un periodo anterior, porque en la práctica también estaría disponible. Normalizamos cada tienda por su demanda media **en train**.

In [ ]:
LOOKBACK, HORIZON = 28, 7
train_end, val_end = int(0.70 * n_days), int(0.85 * n_days)
store_level = demand[:, :train_end].mean(axis=1, keepdims=True)
scaled = demand / store_level


def make_windows(first_day, last_day):
    histories, futures, targets, origins = [], [], [], []
    for store in range(n_stores):
        for t in range(max(first_day, LOOKBACK), last_day - HORIZON + 1):
            histories.append(np.stack([scaled[store, t - LOOKBACK:t], promo[store, t - LOOKBACK:t],
                                       holiday[t - LOOKBACK:t]]))
            futures.append(np.concatenate([promo[store, t:t + HORIZON], holiday[t:t + HORIZON]]))
            targets.append(scaled[store, t:t + HORIZON])
            origins.append((store, t))
    return (np.array(histories, dtype=np.float32), np.array(futures, dtype=np.float32),
            np.array(targets, dtype=np.float32), np.array(origins))


windows = {'train': make_windows(0, train_end), 'val': make_windows(train_end, val_end),
           'test': make_windows(val_end, n_days)}
for split, (hist, fut, target, origin) in windows.items():
    print(f'{split:5s} historia {hist.shape} | futuro {fut.shape} | objetivo {target.shape}')

**Resultado e interpretación.** Cada ejemplo tiene 3 canales de historia (demanda normalizada, promoción y feriado) durante 28 días, 14 covariables futuras (7 promociones y 7 feriados) y 7 valores objetivo. Las ventanas consecutivas se solapan, así que no son independientes: el número de ventanas sobreestima la cantidad real de información.

### Métricas y baselines
Medimos en unidades originales: MAE, RMSE y WAPE $=\sum|y-\hat y|/\sum y$. Los baselines son el **ingenuo estacional** (el mismo día de la semana anterior) y la **media móvil** de 28 días.

In [ ]:
def to_units(values, origin):
    return values * store_level[origin[:, 0], 0][:, None]


def forecast_metrics(y_true, y_pred):
    error = y_pred - y_true
    return {'MAE': round(float(np.abs(error).mean()), 2), 'RMSE': round(float(np.sqrt((error ** 2).mean())), 2),
            'WAPE': round(float(np.abs(error).sum() / y_true.sum()), 4)}


def baseline_forecasts(split):
    hist, _, _, origin = windows[split]
    seasonal_naive = to_units(hist[:, 0, -7:], origin)  # t-7 ... t-1 predicen t ... t+6
    moving_average = to_units(np.repeat(hist[:, 0, :].mean(axis=1, keepdims=True), HORIZON, axis=1), origin)
    return {'ingenuo_estacional': seasonal_naive, 'media_movil_28': moving_average}


y_val_units = to_units(windows['val'][2], windows['val'][3])
for name, forecast in baseline_forecasts('val').items():
    print(f'{name:20s} validación: {forecast_metrics(y_val_units, forecast)}')

**Resultado e interpretación.** El ingenuo estacional captura el patrón semanal sin aprender nada, por lo que suele ser difícil de superar. La media móvil ignora el día de la semana. Ninguno usa las promociones planificadas: ahí está la oportunidad para los modelos con covariables futuras.

### Tres arquitecturas con la misma interfaz
Todas reciben `history` con forma (batch, canales, tiempo) y `future` con las covariables conocidas, y devuelven los 7 días. Solo cambia cómo se resume la historia.

In [ ]:
class MLPForecaster(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3 * LOOKBACK + 2 * HORIZON, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, HORIZON))

    def forward(self, history, future):
        return self.net(torch.cat([history.flatten(1), future], dim=1))


class CNNForecaster(nn.Module):
    def __init__(self, channels=16, hidden=64):
        super().__init__()
        self.conv = nn.Sequential(  # dilataciones 1, 2 y 4: el campo receptivo cubre más de dos semanas
            nn.Conv1d(3, channels, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=2, dilation=2), nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=4, dilation=4), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(channels * LOOKBACK + 2 * HORIZON, hidden), nn.ReLU(),
                                  nn.Linear(hidden, HORIZON))

    def forward(self, history, future):
        return self.head(torch.cat([self.conv(history).flatten(1), future], dim=1))


class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=32, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden_size + 2 * HORIZON, hidden), nn.ReLU(),
                                  nn.Linear(hidden, HORIZON))

    def forward(self, history, future):
        _, (h_n, _) = self.lstm(history.transpose(1, 2))  # (batch, tiempo, canales)
        return self.head(torch.cat([h_n[-1], future], dim=1))


architectures = {'mlp': MLPForecaster, 'cnn_1d': CNNForecaster, 'lstm': LSTMForecaster}
for name, cls in architectures.items():
    print(f'{name:7s} parámetros: {sum(p.numel() for p in cls().parameters())}')

**Resultado e interpretación.** La MLP no sabe que los 28 valores están ordenados en el tiempo; la CNN asume patrones locales que se repiten; la LSTM asume que la historia se puede resumir secuencialmente. Ese **sesgo inductivo** es lo que comparamos, con un número de parámetros del mismo orden.

### Entrenar con el mismo protocolo
Misma semilla, optimizador, pérdida L1 (alineada con MAE), tamaño de batch y *early stopping* en validación para las tres arquitecturas.

In [ ]:
def tensors(split):
    hist, fut, target, _ = windows[split]
    return (torch.tensor(hist, device=device), torch.tensor(fut, device=device), torch.tensor(target, device=device))


data = {split: tensors(split) for split in windows}


def train_forecaster(build_model, epochs=60, lr=3e-3, batch_size=128, patience=8):
    torch.manual_seed(SEED)
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(*data['train']), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    history_val, best_loss, best_state, wait = [], np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for hist, fut, target in loader:
            loss = F.l1_loss(model(hist, fut), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.l1_loss(model(*data['val'][:2]), data['val'][2]).item()
        history_val.append(val_loss)
        if val_loss < best_loss - 1e-5:
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, history_val


def forecast(model, split):
    model.eval()
    with torch.no_grad():
        prediction = model(*data[split][:2]).cpu().numpy()
    return to_units(prediction, windows[split][3])


forecasters, curves, durations = {}, {}, {}
for name, cls in architectures.items():
    start = time.perf_counter()
    forecasters[name], curves[name] = train_forecaster(cls)  # la clase actúa como constructor
    durations[name] = time.perf_counter() - start
    print(f'{name:7s} {len(curves[name])} épocas | {durations[name]:.1f} s')

for name, curve in curves.items():
    plt.plot(curve, label=name)
plt.xlabel('época')
plt.ylabel('L1 validación (escala normalizada)')
plt.legend()
plt.show()

**Resultado e interpretación.** Revisa cuántas épocas necesitó cada modelo y su costo en segundos. Una arquitectura que converge a un error similar en menos tiempo es preferible en producción, donde estos modelos se reentrenan con frecuencia.

### Comparar en validación y evaluar el elegido en test
Reunimos baselines y redes en una tabla de validación, elegimos por WAPE y evaluamos solo el elegido en test, junto al ingenuo estacional como referencia.

In [ ]:
validation = {name: forecast_metrics(y_val_units, f) for name, f in baseline_forecasts('val').items()}
validation.update({name: forecast_metrics(y_val_units, forecast(model, 'val')) for name, model in forecasters.items()})
for name, metrics in sorted(validation.items(), key=lambda item: item[1]['WAPE']):
    print(f'{name:20s} validación: {metrics}')

chosen = min(forecasters, key=lambda name: validation[name]['WAPE'])
y_test_units = to_units(windows['test'][2], windows['test'][3])
test_forecast = forecast(forecasters[chosen], 'test')
print(f'\nRed elegida por validación: {chosen}')
print('test:', forecast_metrics(y_test_units, test_forecast))
print('ingenuo estacional en test:', forecast_metrics(y_test_units, baseline_forecasts('test')['ingenuo_estacional']))

**Resultado e interpretación.** La pregunta relevante es cuánto mejora la red elegida sobre el ingenuo estacional en test, no cuál red gana por décimas en validación. Si la mejora es pequeña, el costo de operar una red (reentrenamiento, monitoreo, explicabilidad) puede no justificarse.

### Error por horizonte y días con promoción
Analizamos dónde se equivoca cada modelo: el error suele crecer con el horizonte, y las promociones son el caso donde las covariables futuras deberían marcar diferencia.

In [ ]:
fut_test = windows['test'][1]
promo_mask = fut_test[:, :HORIZON].astype(bool)
candidates = {'ingenuo_estacional': baseline_forecasts('test')['ingenuo_estacional'],
              **{name: forecast(model, 'test') for name, model in forecasters.items()}}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for name, prediction in candidates.items():
    absolute_error = np.abs(prediction - y_test_units)
    axes[0].plot(range(1, HORIZON + 1), absolute_error.mean(axis=0), marker='o', label=name)
    print(f'{name:20s} MAE en días con promoción: {absolute_error[promo_mask].mean():6.2f} '
          f'| sin promoción: {absolute_error[~promo_mask].mean():6.2f}')
axes[0].set_xlabel('días hacia adelante')
axes[0].set_ylabel('MAE (unidades)')
axes[0].set_title('Error por horizonte (test)')
axes[0].legend()

store, origin_day = 2, windows['test'][3][windows['test'][3][:, 0] == 2][10, 1]
row = np.flatnonzero((windows['test'][3][:, 0] == store) & (windows['test'][3][:, 1] == origin_day))[0]
context = np.arange(origin_day - LOOKBACK, origin_day)
target_days = np.arange(origin_day, origin_day + HORIZON)
axes[1].plot(context, demand[store, context], color='gray', label='historia')
axes[1].plot(target_days, y_test_units[row], 'k-o', label='real')
for name in ('ingenuo_estacional', chosen):
    axes[1].plot(target_days, candidates[name][row], '--o', label=name)
for day in target_days[promo[store, target_days]]:
    axes[1].axvspan(day - 0.5, day + 0.5, color='orange', alpha=0.2)
axes[1].set_title(f'Tienda {store}: un pronóstico de test (naranja = promoción)')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Resultado e interpretación.** Si la red reduce el error sobre todo en días con promoción, su ventaja proviene de usar información del futuro planificado, no de una mejor lectura de la historia: el valor está en el dato y no solo en la arquitectura. El ingenuo estacional, en cambio, traslada las promociones de la semana pasada a la semana que viene.

## Limitaciones y conclusiones clave
La serie es sintética y el generador es conocido; en datos reales hay quiebres de stock (la venta observada subestima la demanda), cambios de surtido y eventos no planificados. Una sola partición temporal es frágil: el protocolo recomendado es la validación con origen móvil (*rolling origin*). Los modelos clásicos (ETS, ARIMA) y los de árboles con variables de calendario son referencias que deberían incluirse antes de elegir una red.

- La partición temporal y las ventanas evitan usar información del futuro que no estaría disponible.
- Las covariables conocidas a futuro pueden aportar más que la arquitectura.
- MLP, CNN 1D y LSTM difieren en su sesgo inductivo; se comparan con el mismo protocolo y contra baselines ingenuos.

## Ejercicios progresivos
1. **Guiado:** reemplaza las covariables futuras por ceros en train, validación y test, reentrena la arquitectura elegida y compara el MAE en días con promoción.
2. **Independiente:** cambia `LOOKBACK` a 14 y a 56 días; reporta WAPE de validación y tiempo de entrenamiento de las tres arquitecturas.
3. **Desafío:** convierte la LSTM en un pronóstico probabilístico entrenando el cuantil 0,9 con la pérdida *pinball* $\max(q\,e, (q-1)\,e)$ con $e=y-\hat y$. Verifica en test qué fracción de días queda bajo el pronóstico P90 y discute su uso para fijar stock de seguridad.

## Conexión con el siguiente laboratorio
Los tres laboratorios de este módulo aplican embeddings, redes densas, convoluciones y recurrencia a problemas de negocio. Úsalos como base para un proyecto con datos reales de tu organización, documentando origen, calidad y licencia de los datos.